## DLA CODE OF CONDUCT V2.0

This Code of Conduct defines the principles governing ethical, transparent, and responsible use of Large Language Models (LLMs), online resources, and peer collaboration in the Deep Learning Applications laboratories. This version of the Code of Conduct was refined via a brainstorming session with **ChatGPT Version 5.2** and subsequently adapted to reflect the specific requirements and values of the DLA laboratories. In that spirit, this Code itself models the transparency it expects from you.

***Our goal is not to restrict innovation, but to ensure integrity, accountability, and genuine learning.***

### 1. Transparency in the Use of LLMs and AI Tools

The use of LLMs and AI-assisted tools is permitted — *but it must be transparent*.

* **Explicit Disclosure:** Clearly state if and how LLMs (e.g., ChatGPT, Copilot, Claude, etc.) were used. This includes code generation, debugging, data analysis, experiment design, report writing, or conceptual clarification.
* **Description of Contribution:** Briefly describe what the tool contributed and how you modified, verified, or extended its output.
* **Acknowledgment of Limitations:** Recognize that LLM outputs may contain errors, biases, or non-optimal solutions. You are responsible for verifying correctness, appropriateness, and academic integrity.

***Using AI does not reduce your responsibility for the final result.***

### 2. Proper Attribution and Documentation

Deep learning builds on existing work — responsibly.

* **Attribution:** Properly cite all external resources, including: Code snippets, Tutorials, Documentation, Datasets, Pretrained models, Research papers, and AI-generated content.
* **Reproducibility:** Clearly document tools, libraries, model versions, hyperparameters, and experimental setups so that your work can be reproduced.
* **Clarity of Modifications:** If you adapt external code, explicitly indicate what you changed and why.

***Transparency is a sign of scientific maturity — not weakness.***

### 3. Collaboration and Individual Responsibility

Discussion is encouraged. Copying is not.

* **Collaborative Learning:** You are encouraged to discuss concepts, debugging strategies, and approaches with classmates.
* **Individual Submission:** Your submitted solution must reflect your own understanding and implementation.
* **No Direct Sharing of Solutions:** Do not share complete solutions, trained models, or reports. Do not submit another person's work — or AI-generated work — as your own without meaningful engagement and proper disclosure.

***If you cannot explain your submission, it is not your submission.***

### 4. Accountability and Academic Integrity

You are responsible for everything you submit. Failure to comply with these guidelines may result in review by the course examination commission and can lead to disciplinary measures in accordance with university regulations.

***Integrity is part of your training as a machine learning practitioner.***

### 5. The Spirit of This Code of Conduct

This course prepares you to work in a field where:

* Reproducibility matters
* Ethical considerations matter
* Transparency matters
* Responsible AI use matters

***The purpose of this Code of Conduct is not surveillance — it is professional formation.***

### TL;DR

Use AI; Don’t let AI use you; Be transparent; Cite everything; Do your own thinking.

***If you can’t explain it, you probably shouldn’t submit it.***

l---
---

## Introduction

In this first laboratory we will see a few ways to exploit and adapt pre-trained models to solve new problems. We will start first by downloading and instantiating a new dataset and establishing a stable and reproducible baseline model based on a pre-trained CNN.

---

## Exercise 1 (Warmup): Exploratory Data Analysis and a Stable Baseline

For this laboratory we will work with [The German Traffic Sign Detection Benchmark](https://benchmark.ini.rub.de/) [1]. We will begin, not with *detection*, but with a simpler traffic sign *classification* problem. This has two advantages: (1) the images are *smaller* than in the detection benchmark; and (2) a wrapper for the GTSRB dataset is conveniently included in the `torchvision` library.  

[1] Houben S, Stallkamp J, Salmen J, Schlipsing M, Igel C. Detection of traffic signs in real-world images: The German Traffic Sign Detection Benchmark. In The 2013 International Joint Conference on Neural Networks (IJCNN), 2013.

### Exercise 1.1: Exploratory Data Analysis

A good best practice to adopt in all experimental deep learning projects is thorough *exploratory data analysis*. In this exercise you should instantiate the GTSRB Dataset, inspect some images, and do some statistical analysis of the distribution of data (and metadata). A goal here is to keep an eye out for anything that might be problematic in what is to come.



In [2]:
# Standard imports and aliases.
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
import seaborn as sns
import torchvision.transforms.v2 as T     # Use transforms v2; much more efficient.
from torchvision.datasets import GTSRB
from torch.utils.data import DataLoader, Dataset
from torchvision.models import list_models, get_model
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [3]:

# buona pratica fare un'analisi esplorativa del dataset

# prima cosa da fare se hai una GPU sulla macchina-->definiamo la variabile device
device = "cuda" if torch.cuda.is_available() else "cpu"


# facciamo lo split tra train e test
# all'inizio vediamo senza l'uso della trasformazione in tensori propio per vedere come effettivamente sono salvati

ds_train = GTSRB("_data/.", split="train", download=True) # train
ds_test = GTSRB("_data/.", split="test", download=True) # test

100%|██████████| 187M/187M [00:09<00:00, 20.1MB/s] 
 21%|██        | 18.8M/89.0M [00:02<00:07, 9.06MB/s]


KeyboardInterrupt: 

Debug per vedere se effettivamente uso la GPU

In [ ]:
print(device)

Creiamo una funzione per la visualizzazione

In [ ]:
def visualize(sample: list[torch.Tensor]):
    plt.figure(figsize=(15,20))
    for (idx, (image, label)) in enumerate(sample):
        plt.subplot(10,10,idx+1)
        plt.imshow(image.permute(1, 2, 0))
        plt.title(f"CLS: {label}")
        plt.axis('off')

Vediamo adesso come sono salvati i dati in generale e gli elementi

In [ ]:
# visione generale del dataset caricato
print(ds_train)
print(ds_test)
# visione dell'elemento
print(ds_train[0]) # è un PIL.Image di dimensione 29x30

ds_train[0][0] # vedo l'immagine

# quasi tutti i dataset vengono forniti così immagine + etichetta però nella pratica noi lavoriamo con altri valori (tensori pytorch)

Convertiamo a una struttura dati a noi più affine

In [ ]:
# versione senza warning
transform = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])

ds_train = GTSRB("_data/.", split="train", transform=transform, download=True)
ds_test = GTSRB("_data/.", split="test", transform=transform, download=True)

In [ ]:
ds_train[0][0].shape # [C, H, W] per una questione di efficienza e da tenere in considerazione per il futuro

Utilissimo la visualizzazione di un sottocampione

In [ ]:
# funzione per prendere un sottocampione
def getSample(num_sample: int = 100):
    image_sample = [ds_train[idx] for idx in np.random.choice(range(len(ds_train)), num_sample)]
    return image_sample

In [ ]:
# selezionamo a caso 100 immagini dal training
image_sample = getSample() # lista

# visualizzazione grafica
visualize(image_sample)

# cosa si può notare? Risoluzione diversa (crop per vedere esclusivamente il cartello) no batch di immagini di dimensioni diverse

Proviamo a raccogliere delle statistiche: distribuzione altezze, larghezze, classi

Sbilanciato o no?

In [ ]:
# utile convertire le statistiche in un dataframe (training)
df_stats = pd.DataFrame([(label, image.shape[1], image.shape[2]) for (image, label) in ds_train], columns=["CLS", "HEIGHT", "WIDTH"])
df_stats["AR"] = df_stats["HEIGHT"] / df_stats["WIDTH"] # colonna aspetto-ratio

In [ ]:
# visualizziamo la descrizione del dataframe
df_stats.describe()

## Osservazioni
cosa osserviamo dalla tabella?

- Altezza e Larghezza hanno una varianza altina
- Altezza e Larghezza hanno la stessa media
- discrepanza tra Altezza e Larghezza minima e massima
- osservando i quantili abbiamo delle immagini outlier molto grosse

Stampiamo dei grafici

In [ ]:
_ = df_stats.hist()

così vediamo la vera distribuzione: notiamo che la concentrazione delle immagini si trova prevelentemente prima di 50 per altezza e lunghezza, classi non perfettamente bilanciate e non troppo sbilanciate (tanti valori per le classi) quindi non è il caso di trattarlo come un caso sbilanciato.

In [ ]:
_ = df_stats.boxplot()

I BoxPlot forniscono l'idea di avere eccessivi outlier

In [ ]:
plt.figure(figsize=(10, 8))
# Calcoliamo la correlazione e creiamo il grafico
sns.heatmap(df_stats.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Analisi Correlazione Dimensioni Immagini")
plt.show()

l'ho fatto solo per curiosità sul fare le heatmap di correlazione non ci fornisce informazioni aggiuntive

Come gestiamo la differenza di dimensioni nel dataset?

In [ ]:
# possibile soluzione
# crop (70) in modo che la dimensione più piccola sia 70, poi un random crop (64, 64) così ci assicura le stesse dimensioni e un'invarianza rispetto alla traslazione e normalizzarlo con i valori di imageNet (resNet) se non uso resNet devo modificare quei valori
transform = T.Compose([T.Resize(70), T.RandomCrop((64, 64)), T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)]), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
ds_train = GTSRB("_data/.", split="train", transform=transform, download=True)
ds_test = GTSRB("_data/.", split="test", transform=transform, download=True) # non mi torna che facciamo le trasformazioni anche al test

In [ ]:
# visualizzazione con le trasformazioni
image_sample = getSample()
visualize(image_sample)

### Analysis

Deep Learning is very much an *experimental* discipline. Experiments are *nothing* without analysis and interpretation. Be sure to **always** stop and analyze the results of preliminary explorations. Note anything significant and -- importantly -- anything that is going to be relevant for what comes next.

So... In this Markdown cell you should collect and report (using, for the love of God, the *rich markup capabilities of Markdown*) any relevant findings you have made before proceding.

**Important Warning**: This is the **one and only** time I will remind you of the need to provide *analysis* and interpretation of your experimental methodology and results. The responsibility is *yours* to include it elsewhere.


---
### Exercise 1.2: A Stable and Reproducible Baseline

In this exercise you should implement code to use a pretrained network as a *feature extractor* that, instead of *classifying* images in input, should return the *feature representation* from the last layer of the pretrained model before the classifier. These features, extracted from the train set, should be used to train a *classical* model for classification (e.g. an SVM, a Nearest Neighbor, or a Linear Discriminant classifier from Scikit-learn). Evaluate the performance of this baseline model on the features extracted from the test set.

In [ ]:

# variabili globali
batch_size= 1024


Per ottenere tutti i pre trained models di torchvision

In [ ]:
list_models()

In [ ]:
# train dataloader
dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True)



#caricamento modello (default preso da torchivision specifiche)
model = get_model("resnet18", weights="DEFAULT").to(device)

Visualizzazione del modello

In [ ]:
model

In [ ]:
# per usare il modello con tutti i pesi togliendo solo la testa del classificatore
# modo facile facile
model.fc = nn.Identity()
model.to(device)
model


Usiamo il dataloader

In [ ]:
train_feats=[]
train_classes=[]
model.eval()

for(images, labels) in tqdm(dl_train):
    images = images.to(device)
    with torch.no_grad(): # non aggiungere a nessun grafo di calcolo
        train_feats.append(model(images))
    train_classes.append(labels)
train_feats = torch.vstack(train_feats).cpu()
train_classes = torch.concat(train_classes)

In [ ]:
train_feats.shape, train_classes.shape

In [ ]:
#baseline riproducibile
# SVC a type of SVM (sklearn)
svc = SVC(kernel="linear") # teniamo i valori base eccetto per il tipo di kernel di svm
# addestriamo
svc.fit(train_feats, train_classes)

### SBAGLIATO DA FARE SUL TEST

In [ ]:
print(classification_report(svc.predict(train_feats), train_classes))

 ### Fatto sul test

In [ ]:
# test dataloader
dl_test = DataLoader(ds_test, batch_size=batch_size, shuffle=False)

test_feats=[]
test_classes=[]
model.eval()

for(images, labels) in tqdm(dl_test):
    images = images.to(device)
    with torch.no_grad(): # non aggiungere a nessun grafo di calcolo
        test_feats.append(model(images))
    test_classes.append(labels)
test_feats = torch.vstack(test_feats).cpu()
test_classes = torch.concat(test_classes)


In [ ]:
print(classification_report(svc.predict(test_feats), test_classes))


---
### Exercise 1.3: A Fine-tuning Baseline

In this exercise you should try to *improve* on the stable baseline given by the feature extraction + SVM (or whatever) classifier you produced in the previous lecture. To do this, you should *fine-tune* the ResNet-18 (or whatever model you chose) to solve the new classification task.

To do this, you could proceed by:
1. Loading the ResNet-18 (or whatever) model and replacing the final FC layer (the classifier) with a *new* classifier. This could be a single Linear layer, or could be an MLP.
2. Training the resulting model on the GTSRB dataset for a few epochs.
3. Evaluating the resulting performance.

Some things you should probably consider (especially thinking about the *next* exercise):
+ You should be monitoring not only the loss on the training set, but also a *validation* loss on an *independent* validation set. Split the training set into two datasets: one with, say, 80% of the original training samples, and another with the remaining 20%. You can use this smaller set to monitor performance and check for *overfitting*.
+ Maybe the best strategy is to not fine-tune *all* layers, but only the last few. Think about *selectively* fine-tuning layers of the network.
+ Maybe a *single* linear layer isn't the best option for the classifier. Think about using an MLP instead.

In [ ]:
def train_epoch(model: nn.Module, dl: torch.utils.data.DataLoader , optimizer: torch.optim.Optimizer, epoch ="Unknown", device ="cpu"):
    model.train()
    losses = []
    for (xs, ys) in tqdm(dl, desc=f"Epoch {epoch}", leave=True):
        xs = xs.to(device)
        ys = ys.to(device)
        optimizer.zero_grad()
        out = model(xs)
        loss = F.cross_entropy(out, ys)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return np.mean(losses)

In [ ]:
def evaluate_model(model: nn.Module, dl: torch.utils.data.DataLoader , device="cpu"):
    model.eval()
    preds = []
    gts = []
    with torch.no_grad():
        for (xs, ys) in tqdm(dl, desc=f"Evaluating", leave=False):
            xs = xs.to(device)
            pred = torch.argmax(model(xs), dim=1)
            gts.append(ys)
            preds.append(pred.detach().cpu().numpy())

        #return accuracy score and classification report
        return (accuracy_score(np.hstack(gts), np.hstack(preds))), classification_report(np.hstack(gts), np.hstack(preds), zero_division=0, digits=1)


# creiamo qui una funzione per il conteggio dei parametri in un modello
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
# pipeline non automatizzata

# creazione modello
model = get_model("resnet18", weights="DEFAULT")
#for param in model.parameters():
#    param.requires_grad_(False)
model.fc = nn.Linear(512, 43) # possibile idea
model.to(device)

# dataloader training e testing
batch_size=1024

dl_train = DataLoader(ds_train, batch_size=batch_size, num_workers=4, shuffle=True, pin_memory=True)

dl_test = DataLoader(ds_test, batch_size=batch_size, shuffle=False, num_workers=True)


In [ ]:
epochs = 20
lr=0.001
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

losses = []
for epoch in range(epochs):
    losses.append(train_epoch(model, dl_train, optimizer, epoch+1, device=device))

plt.plot(losses)
(acc, cls) = evaluate_model(model, dl_test, device=device)
print(acc)
print(cls)

---
---
## Exercise 2: Pipeline Consolidation

Consolidate your implementation. When building applications based asd on Deep Learning, you will inevitably need to run many, many experiments. So, it is *always* a good idea to engineer a reproducible pipeline that allows you to run (and re-run) experiments with different hyperparameters. In this exercise you should do exactly this: engineer a deep learning pipeline that encapsulates (at least) training and evaluation so that you can easily and reproducibly run multiple experiments and compare the results.

Some things to think about when engineering this pipeline:
- **Model and Loss (and maybe Optimizer) Abstraction**: An important variable in training deep models is the model (obviously), the loss (somewhat less obvious), and the optimizer (less obvious) used during training. Your pipeline should probably be able to adapt to these changing configurations.
- **Configuration management**: Instead of global variables specializing each cell, thread a configuration object through your code (or use a singleton). If you are planning to use an IDE and not this notebook for the lab, you might consider using a configuration management library like [OmegaConf](https://omegaconf.readthedocs.io/en/2.3_branch/) which will help instrument your code so that you can pass configuration overrides via command line arguments.
- **Logging**: You will invariably need to run and compare multiple experiments. Instrument your code for monitoring training. Good options are [Tensorboard](https://docs.pytorch.org/tutorials/recipes/recipes/tensorboard_with_pytorch.html) or [Weights and Biases (WandB)](https://wandb.ai/site/). My PhD students seem to overwhelmingly prefer Weights and Biases. The configuration parameters should be used to facilitate distinguishing and comparing multiple runs with multiple hyperparameters.

 Procederò con l'utilizzo del jupyter notebook per la comodità del commento (modulerò come se fosse un python modulo ad eccezione che al posto della config.yaml sarà presente un file o codice python che li gestisce)

## Import utili

In [ ]:
import os
import glob
import logging
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
import seaborn as sns
import torchvision.transforms.v2 as T
from torchvision.datasets import GTSRB
from torch.utils.data import DataLoader, Dataset
from torchvision.models import list_models, get_model
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torchinfo import summary
from rich.console import Console
from rich.logging import RichHandler
import torchvision.utils as vutils
import wandb
import getpass
from types import SimpleNamespace
from ipdb import launch_ipdb_on_exception
import tensorboard
from torch.utils.tensorboard import SummaryWriter

## Visualizzazione
fornisco funzioni per la visualizzazione del modello parametri e visualizzazione dataset

In [ ]:
# stampa la struttura del modello e numero dei parametri
console = Console()
def visualize(model: nn.Module, model_name: str, input_data: torch.Tensor):
    out = model(input_data)
    console.print(f'Computed output, shape = {out.shape=}')
    model_stats = summary(model,
                          input_data=input_data,
                          col_names=[
                              "input_size",
                              "output_size",
                              "num_params",
                              # "params_percent",
                              # "kernel_size",
                              # "mult_adds",
                          ],
                          row_settings=("var_names",),
                          col_width=18,
                          depth=8,
                          verbose=0,
                          )
    console.print(model_stats)

#mostra le immagini di un campione del dataset
def visualize_dt(sample: list[torch.Tensor]):
    plt.figure(figsize=(15,20))
    for (idx, (image, label)) in enumerate(image_sample):
        plt.subplot(10,10,idx+1)
        plt.imshow(image.permute(1, 2, 0))
        plt.title(f"CLS: {label}")
        plt.axis('off')

## Analisi dati
qui sono presenti le funzioni usate nell'es precedente per l'analisi dati del dataset. Di solito essendo di analisi esplorativa di pre-processing non saranno presenti nella pipeline di lavoro

In [ ]:
# funzione per prendere un sottocampione
def getSample(ds_train: torch.utils.data.DataLoader, num_sample: int = 100, ):
    image_sample = [ds_train[idx] for idx in np.random.choice(range(len(ds_train)), num_sample)]
    return image_sample

# funzione per creare un dataframe di pandas da un dataloader (bug se si inserisce più nomi di tre)
def createDataFrame(ds_train: torch.utils.data.DataLoader, column_names: list[str]= ["CLS", "HEIGHT", "WIDTH"], ar: bool= True):
    if len(column_names) == 0 or len(column_names) > 3:
        print("Errore: servono esattamente 1-3 colonne.")
        return None
    df_stats = pd.DataFrame([(label, image.shape[1], image.shape[2]) for (image, label) in ds_train], columns=column_names)
    if ar:
        df_stats["AR"] = df_stats["HEIGHT"] / df_stats["WIDTH"] # colonna aspetto-ratio
    return df_stats

def createGraph(df: pd.DataFrame, type: str="hist"):
    if type == "hist":
        return df.hist()
    elif type == "boxplot":
        return df.boxplot()
    elif type == "violin":
        fig, ax = plt.subplots()
        sns.violinplot(data=df, ax=ax)
        return ax
    elif type == "heatmap":
        plt.figure(figsize=(10, 8))
        # Calcoliamo la correlazione e creiamo il grafico
        sns.heatmap(df_stats.corr(), annot=True, cmap='coolwarm', fmt=".2f")
        plt.title("Analisi Correlazione Dimensioni Immagini")
        plt.show()
    else:
        print("invalid input")




## Dataset
sarà presente solo la classe makedataloader perché non è necessario creare una classe dataset dato che il dataset lo prendiamo direttamente da torchvision

In [ ]:
# funzione che crea i dataloader di training e validation
class MakeDataloaders:
    def __init__(self, opts, data):
        self.opts = opts
        generator = torch.Generator().manual_seed(opts.seed)
        train, val = torch.utils.data.random_split(data, lengths=[1 - opts.val_size, opts.val_size],
                                                    generator=generator)
        self.dl_train = DataLoader(train, batch_size=opts.batch_size, shuffle=True, num_workers= opts.num_workers, pin_memory=True)
        self.dl_val = DataLoader(val, batch_size=opts.batch_size, shuffle=False, pin_memory=True, num_workers=True)

def get_transform(opts): # trasform solo per valori di resnet imagenet
    return T.Compose([T.Resize(opts.resize_size), T.RandomCrop((opts.crop_size, opts.crop_size)), T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)]), T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])



## Model


In [ ]:
from torchvision.ops import MLP

# in realtà funziona esclusivamente con resnet andrebbe da sistemare
def get_pretrained_model(opts):
    model = get_model(opts.model_name, weights="DEFAULT")
    if opts.freeze_model:
        for param in model.parameters():
            param.requires_grad_(False)
    if opts.freeze_specific_layer:
        for name, param in model.named_parameters():
            if opts.freeze_specific_layer in name:  # o qualsiasi nome del layer
                param.requires_grad = False
    model.fc = nn.Sequential(
        nn.Linear(512, 1024),
        nn.ReLU(),
        nn.Linear(1024, 512),
        nn.ReLU(),
        nn.Linear(512, opts.num_classes),
    )
    model.to(opts.device)
    return model

def get_personal_model(opts):
    """Crea un modello personalizzato. Da implementare."""
    raise NotImplementedError("get_personal_model non ancora implementato.")



## Utils
sono presenti qui funzioni che non rientrano in particolari categorie, ma hanno lo scopo di helper (non detto che servano sempre)

In [ ]:
def get_logger():
    """
    Crea e configura un logger con output colorato usando Rich.
    Utile per stampare messaggi formattati durante il training.
    """
    FORMAT = "%(message)s"
    logging.basicConfig(
        level="NOTSET", format=FORMAT, datefmt="[%X]", handlers=[RichHandler()]
    )
    log = logging.getLogger("rich")
    return log


LOG = get_logger()


# def N(x):
#     """
#     Converte un tensore PyTorch in array NumPy.
#     Rimuove il tensore dalla GPU e dal grafo computazionale.
#     Utile per calcoli con NumPy o per stampare valori.
#     """
#     return x.detach().cpu().numpy()


# def save_checkpoint(model, optimizer, scheduler, epoch, loss, opts):
#     """
#     Salva un checkpoint del modello durante il training.
#
#     Salva:
#     - Pesi del modello (model.state_dict)
#     - Stato dell'optimizer (learning rate, momenti, ecc.)
#     - Numero dell'epoca corrente
#     - Valore della loss
#
#     Questo permette di riprendere il training in caso di interruzione.
#     """
#     fname = os.path.join(opts.checkpoint_dir, f'e_{epoch:05d}.chp')
#     info = dict(model_state_dict=model.state_dict(),
#                 optimizer_state_dict=optimizer.state_dict(),
#                 scheduler_state_dict=scheduler.state_dict() if scheduler is not None else None,
#                 epoch=epoch, loss=loss, global_step=0)
#     torch.save(info, fname)
#     LOG.info(f'Saved checkpoint {fname}')
#
#
# def load_checkpoint(model, optimizer, scheduler, opts, epoch=None, checkpoint_path=None):
#     if checkpoint_path is not None:
#         fname = checkpoint_path
#     elif epoch is not None:
#         fname = os.path.join(opts.checkpoint_dir, f'e_{epoch:05d}.chp')
#     else:
#         chk_files = glob.glob(os.path.join(opts.checkpoint_dir, "*.chp"))
#         if not chk_files:
#             LOG.warning(" Nessun checkpoint trovato.")
#             return None
#         chk_files.sort()
#         fname = chk_files[-1]
#
#     LOG.info(f" Caricamento checkpoint: {fname}")
#
#     checkpoint = torch.load(fname, map_location='cpu')
#
#     model.load_state_dict(checkpoint['model_state_dict'])
#
#     if optimizer is not None and 'optimizer_state_dict' in checkpoint:
#         optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
#
#         # Sposta stato optimizer su GPU
#         for state in optimizer.state.values():
#             for k, v in state.items():
#                 if isinstance(v, torch.Tensor):
#                     state[k] = v.to(opts.device)
#
#     if scheduler is not None and 'scheduler_state_dict' in checkpoint:
#         scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
#         # Lo scheduler NON ha bisogno di essere spostato su GPU
#
#     model.to(opts.device)
#
#     loaded_epoch = checkpoint['epoch']
#     LOG.info(f" Checkpoint caricato! (Epoca: {loaded_epoch})")
#
#     return checkpoint

## CUDA Check

In [ ]:
print("PyTorch loaded from:", torch.__file__)
print("NumPy loaded from:", np.__file__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Torch version: ", torch.__version__)

## WANDB
ho riscontrato diversi problemi nell'utilizzo di wandb e quindi sono passato a tensorboard

### Login a wandb

In [ ]:
#api_key = getpass.getpass("Inserisci la tua WandB API key: ")
#wandb.login(key=api_key)

### Inizializzazione run

In [ ]:


# # da valutare se salvare tutti questi dati o se devo salvarne altri
# def defineRun(opts):
#     config = {
#         "learning_rate": opts.learning_rate,
#         "model-name": opts.model_name,
#         "dataset": opts.dataset,
#         "epochs": opts.epochs,
#         "batch_size": opts.batch_size,
#         "optimizer": opts.optimizer,
#         "scheduler": opts.scheduler,
#         "loss": opts.loss,
#     }
#
#     if opts.momentum and opts.weight_decay: # estendibile
#         config["momentum"] = opts.momentum
#         config["weight_decay"] = opts.weight_decay
#
#     run = wandb.init(
#         entity="lorenzomariapennelli-",
#         project="lab1",
#         config=config,
#     )
#     return run

## Tensorboard

### inizializzazione run

In [ ]:

def defineRun(opts):
    """
    Crea una SummaryWriter di TensorBoard.
    Sostituisce wandb.init().
    I log vengono salvati in runs/<model_name>_<optimizer>_lr<lr>
    """
    run_name = f"{opts.model_name}_{opts.optimizer}_lr{opts.learning_rate}"
    log_dir = os.path.join("runs", run_name)
    writer = SummaryWriter(log_dir=log_dir)

    # Logga gli iperparametri come testo
    hparams = {
        "learning_rate": opts.learning_rate,
        "model_name":    opts.model_name,
        "dataset":       opts.dataset,
        "epochs":        opts.epochs,
        "batch_size":    opts.batch_size,
        "optimizer":     opts.optimizer,
        "scheduler":     opts.scheduler,
        "loss":          opts.loss,
        "weight_decay":  opts.weight_decay if opts.weight_decay is not None else 0,
        "gamma":         opts.gamma,
    }
    writer.add_text("hparams", str(hparams), global_step=0)
    LOG.info(f"TensorBoard logging in: {log_dir}")
    return writer

## Training

qui verrà inserito il loop di training e valutazione delle prestazioni del modello sia in training che in validazione

In [ ]:
def test_metrics(model, val, opts): # calcolo l'accuratezza
    # 1. Imposta il modello in modalità valutazione
    model.eval()

    correct = []

    # 2. Disabilita il calcolo dei gradienti
    with torch.no_grad():
        for Xcpu, Y in val:
            X = Xcpu.to(opts.device)
            Y = Y.to(opts.device)
            out = model(X)

            # Calcolo dell'accuratezza (corretto per Label Encoding)
            # np.argmax(N(out), axis=1) -> Indice predetto
            # N(Y) -> Indice vero
            predictions = out.argmax(dim=1)

            c = (predictions == Y).cpu().tolist()
            correct.extend(c)

    return np.mean(correct)

def train_loop(model, train, test, opts):
    run=defineRun(opts) # definisco la sessione per il mio training loop

    # Weight decay di default
    if opts.weight_decay is None:
        weight_decay = 0.001 if opts.optimizer == "AdamW" else 0
    else:
        weight_decay = opts.weight_decay

    momentum = opts.momentum if opts.momentum is not None else 0

    # optimizer (questa parte dipende da quanti optimizer vogliamo testare e che parametri)
    if opts.optimizer == "SGD":
        optimizer = torch.optim.SGD(
            model.parameters(), lr=opts.learning_rate,
            momentum=momentum, weight_decay=weight_decay
        )
    elif opts.optimizer == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(), lr=opts.learning_rate,
            weight_decay=weight_decay
        )
    elif opts.optimizer == "AdamW":
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=opts.learning_rate,
            weight_decay=weight_decay
        )
    else:
        raise ValueError(f"Optimizer non valido: {opts.optimizer}")


    # scheduler
    # stesso concetto if in base a quanti scheduler testare
    # per una questione di semplicità ci sarà scritto solo uno
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=opts.gamma)

    #funzioni di loss
    #si basa sempre al contesto dato che il nostro obiettivo è la classificazione solo cross entropy
    loss_fn = torch.nn.CrossEntropyLoss()
    step = 0

    start_epoch = 1
    global_step = 0

    # tralasciamo per ora la precisione mista grazie a gradscaler()

    # if opts.resume: # caricare i checkpoint
    #     checkpoint = load_checkpoint(model, optimizer, scheduler, opts)
    #     if checkpoint is not None:
    #         global_step = checkpoint.get('global_step', 0)
    #         start_epoch = checkpoint['epoch'] + 1

    step = global_step

    LOG.info(f" Training da epoca {start_epoch} a {opts.epochs} | Step: {step}")
    for epoch in range(start_epoch, opts.epochs + 1):
        model.train()
        losses = []
        correct = []
        batch_idx = 0
        for Xcpu, Y in train:
            batch_idx += 1
            X = Xcpu.to(opts.device)
            Y = Y.to(opts.device)
            optimizer.zero_grad()
            out = model(X)
            loss = loss_fn(out, Y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            c = (out.detach().argmax(dim=1) == Y).float().mean().item() # altrimenti N() o mantengo tutto su pytorch
            correct.append(c)
            if batch_idx % opts.log_every == 0:
                train_l = np.mean(losses[-opts.batch_window:])
                train_a = np.mean(correct[-opts.batch_window:])
                test_a = test_metrics(model, test, opts)
                msg = f'{epoch:03d}.{batch_idx:03d}: '
                msg += f'train: loss={train_l:1.6f}, acc={train_a:1.3f} '
                msg += f'test: acc={test_a:1.3f}'
                LOG.info(msg)

                # run.log({
                #     "train/loss": train_l,
                #     "train/accuracy": train_a,
                #     "test/accuracy": test_a,
                #     "learning_rate": optimizer.param_groups[0]['lr'],
                #     "epoch": epoch,
                #     "step": step,
                # })
                # ── TensorBoard logging (sostituisce run.log) ──
                run.add_scalar('train/loss',     train_l, step)
                run.add_scalar('train/accuracy', train_a, step)
                run.add_scalar('val/accuracy',  test_a,  step)
                run.add_scalar('learning_rate',  optimizer.param_groups[0]['lr'], step)
                step += 1

        # if epoch % opts.save_every == 0:
        #     save_checkpoint(model, optimizer, scheduler, epoch, loss, opts)
        scheduler.step()
        LOG.info(f"Epoch {epoch} | LR: {scheduler.get_last_lr()}")

    #run.finish()
    run.close()



def main(opts):
    # Parametri del modello
    input_size = opts.image_size  # Dimensione immagini (224x224 pixel)
    input_data = torch.randn(opts.batch_size, 3, input_size, input_size).to(opts.device)  # Dati fake per visualizzazione
    num_classes = opts.num_classes

    # Crea modello
    if opts.personalized:
        model = get_personal_model(opts)
    else:
        model = get_pretrained_model(opts)

    # Visualizza architettura (salva grafo del modello)
    visualize(model, opts.model_name, input_data)

    # Sposta modello su GPU se disponibile
    model = model.to(opts.device)

    transform = get_transform(opts)

    ds_train = GTSRB("_data/.", split="train", transform=transform, download=True)

    dec = MakeDataloaders(opts, ds_train)
    train = dec.dl_train
    val = dec.dl_val

    ds_test = GTSRB("_data/.", split="test", transform=transform, download=True)
    # test dataloader
    dl_test = DataLoader(ds_test, batch_size=opts.batch_size, shuffle=False)

    # Avvia training
    train_loop(model, train, val, opts)





## Configurazione

## MAIN PIPELINE


In [ ]:

config = {
    # ── Modello ──────────────────────────────
    "model_name":       "resnet18",   # nome del modello torchvision
    "personalized":     False,        # True = usa get_personal_model()
    "num_classes":      43,           # GTSRB ha 43 classi
    "freeze_model":     False,        # freeze di tutto il modello
    "freeze_specific_layer": None,    # freeze di uno specifico layer
    "hidden_channels": [256, 128],  # layer nascosti del MLP
    "mlp_dropout": 0.1,             # dropout rate
    # ── Immagini ─────────────────────────────
    "image_size":       64,          # dimensione input del modello
    "resize_size":      70,          # resize prima del crop
    "crop_size":        64,          # dimensione del random crop

    # ── Dataset / Split ──────────────────────
    "dataset":          "GTSRB",
    "val_size":         0.2,          # frazione di train usata per validation
    "seed":             1234,

    # ── Dataloader ───────────────────────────
    "batch_size":       1024,
    "num_workers":      4,

    # ── Ottimizzatore ────────────────────────
    "optimizer":        "AdamW",      # "SGD" | "Adam" | "AdamW"
    "learning_rate":    2e-4,
    "weight_decay":     5e-4,
    "momentum":         None,         # usato solo con SGD

    # ── Scheduler ────────────────────────────
    "scheduler":        "ExponentialLR",
    "gamma":            0.95,         # decay factor per ExponentialLR

    # ── Loss ─────────────────────────────────
    "loss":             "CrossEntropyLoss",

    # ── Training ─────────────────────────────
    "epochs":           10,
    "log_every":        5,           # log ogni N batch
    "batch_window":     5,           # finestra mobile per media loss/acc
    "save_every":       5,            # salva checkpoint ogni N epoche
    "checkpoint_dir":   "checkpoints/resnet18",
    "resume":           False,        # True = riprendi da ultimo checkpoint
}

In [ ]:
# Converte dict config in oggetto SimpleNamespace
opts = SimpleNamespace(**config)

# Usa GPU se disponibile, altrimenti CPU
opts.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# launch_ipdb_on_exception: se c'è un errore, apre debugger invece di crashare
with launch_ipdb_on_exception():
     main(opts)

## Visualizzazione tensorboard (only jupyter)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## Testing

In [ ]:
# inference functions
def inference_model(model: nn.Module, dl: torch.utils.data.DataLoader , device="cpu", checkpoint_path=None):
    # if checkpoint_path:
    #     model = load_checkpoint(model, optimizer=None, scheduler=None,
    #                             opts=SimpleNamespace(device=device, checkpoint_dir=os.path.dirname(checkpoint_path)),
    #                             checkpoint_path=checkpoint_path)
    model.eval()
    preds = []
    gts = []
    with torch.no_grad():
        for (xs, ys) in tqdm(dl, desc=f"Evaluating", leave=False):
            xs = xs.to(device)
            pred = torch.argmax(model(xs), dim=1)
            gts.append(ys)
            preds.append(pred.detach().cpu().numpy())

        #return accuracy score and classification report
        return (accuracy_score(np.hstack(gts), np.hstack(preds))), classification_report(np.hstack(gts), np.hstack(preds), zero_division=0, digits=1)



## Pipeline Testing

In [ ]:
# semplici funzioni di testing

(acc, cls)=inference_model(model, dl_test, opts.device, opts.checkpoint_dir)
print(acc)
print(cls)


---
---
## Exercise 3: Choose your Own Adventure

As promised, you should choose **one** of the following exercises to work. Well, at *least* one. If you want to do them all, that is also OK! 

---
### Exercise 3.1 (Easy): Improving Fine-tuning Performance

In this exercise you are asked to iterate on the fine-tuning experiment performed in Exercise 1.3 in order to squeeze the best performance possible out of the model.

What can we do:
- Use a more powerful model?
- More aggressive data augmentation?
- Selective layer training?
- Something else?

**Why choose this exercise?** To hone your skills at incrementally improving fined-tuned model performance via a sequence of (carefully monitored) experiments.

In [ ]:
# Your code here.


---
### Exercise 3.2: Retrieval as Training-free Classification (a bit harder)

In this exercise you will treat the problem of classifying road signs as a type of retrieval problem. By doing so, will will avoid the need to perform extensive fine-tuning of our model and will instead count of the good *representations* provided by massively pretrained models.

How, you ask? In this exercise you should treat the training set as a *gallery* of indexed image descriptors and the test set as a set of *query* descriptors. You could proceed in this way:

1. **Implement** a generic *feature extraction* function that, given a model and a dataloader, will extract the feature representations of all images in the dataloader. You should make this function generic because you will of course want to try *multiple* pretrained backbones in order to pick the best one.
2. **Implement** a function to *query* the gallery using all extracted extracted from the test set. The results of this should be a *ranking* of all gallery images in terms of similarity with each test image. You should think carefully about how to compute this *similarity* score and how to compute it efficiently.
3. **Evaluate** the retrieval performance on each of the 43 classes in the GTSRB dataset using the similarity scores. This could be done using precision recall curves, average precision/recall, or other metrics.

Wait, this isn't a classifier... Well, no. But, if you use the above *pipeline* to select the best feature extraction pipeline, you can then implement a **Nearest-Mean Classifier (NMC)**: compute the *mean* feature representation of all training images for each class and *classify* test images based on which mean is **nearest** to it.

**Why choose this exercise?** To learn how to make the most out of pretrained feature extraction backbones, how to evaluate different types of learning systems (retrieval in this case), and to familiarize yourself with similarity-based classification which is extensively used in multimodal models like CLIP.


---
### Exercise 3.3: *Detecting* Traffic Signs (hardest)

In this exercise you will see if you can take your pretrained *classification* backbone and turn it into a *detector*. We will need another dataset for this -- one with *full image frames* instead of cropped sign images. Luckily, this dataset is available on [Hugging Face](https://huggingface.co/datasets/keremberke/german-traffic-sign-detection) (see below for how to access it).

For this exercise, you could:
1. Start from an available Faster-RCNN model from `torchvision` and *replace* the feature extraction backbone with your ResNet fine-tuned on GTRSB.
2. This *stitched-together* network should do an OK-ish job already of detecting at least *some* traffic signs. See how it performs on some of the images from the GTRSB Detection dataset. Implement a simple function to visualize detections superimposed over the original image. Compute some detection metrics (accuracy @ IoU=0.5, for example).
3. If you're feeling ambitious, you could also *fine-tune* the resulting model to improve detection performance.

**NOTE**: The pretrained Faster-RCNN models in `torchvision` are all based on ResNet-50, so this exercise will likely be much less painful if you use a ResNet-50 from the very start.

**Also Note**: To use the **GTRSB Detection Dataset** you should use the `datasets` package from Hugging Face (version 3.X):

     # If using uv.
     uv add datasets==3

or

     # If using Anaconda
     conda install -c conda-forge datasets==3

In [ ]:
#from datasets import load_dataset

# Download the full detection dataset from the Hugging Face Hub.
# You will want to spend some time studying the organization of this dataset.
#ds = load_dataset("keremberke/german-traffic-sign-detection", name="full")

# Your code here.

---
---